In [ ]:
import xarray as xr
import rasterio
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable

In [ ]:
template_ds = xr.open_zarr("../data_working/treemap2016_hostba_hydro.zarr/", chunks={})
bounds = template_ds.rio.transform_bounds(4326)

subsetter = dict(
    time=slice("1999", "2023"),
    lon=slice(bounds[0], bounds[2]),
    lat=slice(bounds[3], bounds[1])
)

In [ ]:
vars = ["def", "tmin", "vpd"]

aggs = [xr.DataArray.sum, xr.DataArray.min, xr.DataArray.max]

In [ ]:
def process_ds(var: str, agg: Callable) -> xr.Dataset:
    link = f"http://thredds.northwestknowledge.net:8080/thredds/dodsC/agg_terraclimate_{var}_1958_CurrentYear_GLOBE.nc"
    ds = xr.open_dataset(
        link,
        chunks={}
    ).sel(**subsetter)\
        .compute()\
        .resample(time="1YE")\
        .apply(agg, dim="time")\
        .drop_vars("crs")\
        .rename(lon="x", lat="y")\
        .rio.write_crs(4326)\
        .rio.reproject_match(template_ds, resampling=rasterio.enums.Resampling.bilinear)
        
    return ds

In [ ]:
ds_by_var = [
    process_ds(v, a)
    for (v,a) in zip(vars, aggs)
]

In [ ]:
xr.combine_by_coords(ds_by_var).to_zarr("../data_working/terraclimate.zarr")